In [5]:
import sqlite3
import random
import string
import time
import math

In [6]:
conn = sqlite3.connect("students_demo.db")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS students (
        student_id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        gpa REAL NOT NULL
    )
""")

cursor.execute("""CREATE INDEX IF NOT EXISTS idx_gpa ON students(gpa)""")

conn.commit()
print("Schema and indexes created successfully")

Schema and indexes created successfully


In [7]:
def random_name():
    first = ''.join(random.choices(string.ascii_uppercase, k=1)) + ''.join(random.choices(string.ascii_lowercase, k=5))
    last = ''.join(random.choices(string.ascii_uppercase, k=1)) + ''.join(random.choices(string.ascii_lowercase, k=7))
    return f"{first} {last}"

students = [(i, random_name(), round(random.uniform(2.0, 4.0), 2)) for i in range(1, 10001)]

cursor.executemany("INSERT INTO students VALUES (?, ?, ?)", students)
conn.commit()
print(f"Inserted {len(students)} students.")

Inserted 10000 students.


In [8]:
start = time.time()
cursor.execute("SELECT * FROM students WHERE student_id = 5000")
result = cursor.fetchone()
end = time.time()

print(f"Result: {result}")
print(f"Time: {(end - start) * 1000:.4f} ms")

Result: (5000, 'Qewdok Qyjofguo', 2.34)
Time: 5.0642 ms


In [9]:
start = time.time()
cursor.execute("SELECT * FROM students WHERE gpa BETWEEN 3.5 AND 3.8")
results = cursor.fetchall()
end = time.time()

print(f"Rows returned: {len(results)}")
print(f"Time: {(end - start) * 1000:.4f} ms")

Rows returned: 1562
Time: 2.7447 ms


In [10]:
cursor.execute("SELECT name, type FROM sqlite_master WHERE type='index'")
print(cursor.fetchall())

[('idx_gpa', 'index')]


In [11]:
# Disable index by using a function trick that forces full table scan
start = time.time()
cursor.execute("SELECT * FROM students WHERE student_id + 0 = 5000")
result = cursor.fetchone()
end = time.time()

print(f"Result: {result}")
print(f"Linear scan time: {(end - start) * 1000:.4f} ms")

Result: (5000, 'Qewdok Qyjofguo', 2.34)
Linear scan time: 2.0759 ms


In [12]:
times_index = []
times_linear = []

for student_id in random.sample(range(1, 10001), 100):
    # Index scan
    start = time.time()
    cursor.execute("SELECT * FROM students WHERE student_id = ?", (student_id,))
    cursor.fetchone()
    times_index.append((time.time() - start) * 1000)

    # Linear scan
    start = time.time()
    cursor.execute("SELECT * FROM students WHERE student_id + 0 = ?", (student_id,))
    cursor.fetchone()
    times_linear.append((time.time() - start) * 1000)

print(f"Avg index scan:  {sum(times_index)/len(times_index):.4f} ms")
print(f"Avg linear scan: {sum(times_linear)/len(times_linear):.4f} ms")
print(f"Speedup: {sum(times_linear)/sum(times_index):.2f}x")

Avg index scan:  0.1990 ms
Avg linear scan: 1.4698 ms
Speedup: 7.39x


In [13]:
print("Point lookup (index scan)")
cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM students WHERE student_id = 5000")
for row in cursor.fetchall():
    print(row)

print("Range query on gpa (index scan)")
cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM students WHERE gpa BETWEEN 3.5 AND 3.8")
for row in cursor.fetchall():
    print(row)

print("Forced linear scan")
cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM students WHERE student_id + 0 = 5000")
for row in cursor.fetchall():
    print(row)

Point lookup (index scan)
(2, 0, 0, 'SEARCH students USING INTEGER PRIMARY KEY (rowid=?)')
Range query on gpa (index scan)
(3, 0, 0, 'SEARCH students USING INDEX idx_gpa (gpa>? AND gpa<?)')
Forced linear scan
(2, 0, 0, 'SCAN students')


## Part 2: B+ Tree Implementation in Python

Building a B+ tree from scratch to demonstrate the data structure internals. We then compare it against SQLite's C-level index and a forced linear scan.

In [15]:
class BPlusNode:
    def __init__(self, is_leaf=False):
        self.keys = []
        self.values = []  # child nodes (internal) or records (leaf)
        self.is_leaf = is_leaf
        self.next = None  # linked list across leaves for range scans
        self.prev = None

class BPlusTree:
    def __init__(self, order=100):
        self.root = BPlusNode(is_leaf=True)
        self.order = order

    def _find_leaf(self, key):
        node = self.root
        while not node.is_leaf:
            i = len(node.keys)
            for j, k in enumerate(node.keys):
                if key < k:
                    i = j
                    break
            node = node.values[i]
        return node

    def search(self, key):
        leaf = self._find_leaf(key)
        for i, k in enumerate(leaf.keys):
            if k == key:
                return leaf.values[i]
        return None

    # def range_scan(self, low, high):
    #     leaf = self._find_leaf(low)
    #     results = []
    #     while leaf is not None:
    #         for i, k in enumerate(leaf.keys):
    #             if k > high:
    #                 return results
    #             if k >= low:
    #                 results.append(leaf.values[i])
    #         leaf = leaf.next
    #     return results

    def range_scan(self, low, high):
        leaf = self._find_leaf(low)
    
        while leaf.prev is not None and leaf.prev.keys and leaf.prev.keys[-1] >= low:
            leaf = leaf.prev
    
        results = []
        while leaf is not None:
            for i, k in enumerate(leaf.keys):
                if k > high:
                    return results
                if k >= low:
                    results.append(leaf.values[i])
            leaf = leaf.next
        return results
    
    def insert(self, key, value):
        path = []
        node = self.root
        while not node.is_leaf:
            i = len(node.keys)
            for j, k in enumerate(node.keys):
                if key < k:
                    i = j
                    break
            path.append((node, i))
            node = node.values[i]

        i = 0
        while i < len(node.keys) and node.keys[i] < key:
            i += 1
        node.keys.insert(i, key)
        node.values.insert(i, value)

        if len(node.keys) > self.order:
            self._split_leaf(node, path)

    def _split_leaf(self, leaf, path):
        mid = len(leaf.keys) // 2
        new_leaf = BPlusNode(is_leaf=True)
        new_leaf.keys = leaf.keys[mid:]
        new_leaf.values = leaf.values[mid:]
        leaf.keys = leaf.keys[:mid]
        leaf.values = leaf.values[:mid]
        new_leaf.next = leaf.next
        
        if leaf.next is not None:
            leaf.next.prev = new_leaf
        new_leaf.prev = leaf
        
        leaf.next = new_leaf
        self._push_up(path, leaf, new_leaf.keys[0], new_leaf)

    def _split_internal(self, node, path):
        mid = len(node.keys) // 2
        promote_key = node.keys[mid]
        new_node = BPlusNode(is_leaf=False)
        new_node.keys = node.keys[mid + 1:]
        new_node.values = node.values[mid + 1:]
        node.keys = node.keys[:mid]
        node.values = node.values[:mid + 1]
        self._push_up(path, node, promote_key, new_node)

    def _push_up(self, path, left, key, right):
        if not path:
            new_root = BPlusNode(is_leaf=False)
            new_root.keys = [key]
            new_root.values = [left, right]
            self.root = new_root
            return
        parent, idx = path.pop()
        parent.keys.insert(idx, key)
        parent.values.insert(idx + 1, right)
        if len(parent.keys) > self.order:
            self._split_internal(parent, path)


print("B+ Tree class defined successfully")

print(f"Tree height for 10,000 records (order=100): {math.ceil(math.log(10000, 100))} levels")


B+ Tree class defined successfully
Tree height for 10,000 records (order=100): 2 levels


In [16]:
# Build B+ trees on student_id and gpa from the existing SQLite data
cursor.execute("SELECT student_id, name, gpa FROM students")
all_students = cursor.fetchall()

start = time.time()
bpt_id = BPlusTree(order=100)
for sid, name, gpa in all_students:
    bpt_id.insert(sid, (sid, name, gpa))
build_time_id = (time.time() - start) * 1000

start = time.time()
bpt_gpa = BPlusTree(order=100)
for sid, name, gpa in sorted(all_students, key=lambda x: x[2]):
    bpt_gpa.insert(gpa, (sid, name, gpa))
build_time_gpa = (time.time() - start) * 1000

print(f"B+ Tree on student_id built in {build_time_id:.2f} ms")
print(f"B+ Tree on gpa built in {build_time_gpa:.2f} ms")
print(f"Total records indexed: {len(all_students)}")


B+ Tree on student_id built in 178.14 ms
B+ Tree on gpa built in 147.78 ms
Total records indexed: 10000


In [17]:
# Point lookup benchmark: SQLite index vs Python B+ Tree vs Linear scan
# Use perf_counter and repeat 1000x for accurate sub-millisecond timings
sample_ids = random.sample(range(1, 10001), 100)

times_sqlite, times_bptree, times_linear = [], [], []
REPS = 1000

for sid in sample_ids:
    t = time.perf_counter()
    for _ in range(REPS):
        cursor.execute("SELECT * FROM students WHERE student_id = ?", (sid,))
        cursor.fetchone()
    times_sqlite.append((time.perf_counter() - t) * 1000 / REPS)

    t = time.perf_counter()
    for _ in range(REPS):
        bpt_id.search(sid)
    times_bptree.append((time.perf_counter() - t) * 1000 / REPS)

    t = time.perf_counter()
    for _ in range(REPS):
        cursor.execute("SELECT * FROM students WHERE student_id + 0 = ?", (sid,))
        cursor.fetchone()
    times_linear.append((time.perf_counter() - t) * 1000 / REPS)

avg_sqlite = sum(times_sqlite) / len(times_sqlite)
avg_bptree = sum(times_bptree) / len(times_bptree)
avg_linear = sum(times_linear) / len(times_linear)

print("Point Lookup (avg over 100 queries x 1000 reps)")
print(f"  Linear scan:    {avg_linear:.4f} ms  (baseline)")
print(f"  Python B+Tree:  {avg_bptree:.4f} ms  ({avg_linear / avg_bptree:.1f}x faster than linear)")
print(f"  SQLite index:   {avg_sqlite:.4f} ms  ({avg_linear / avg_sqlite:.1f}x faster than linear)")


Point Lookup (avg over 100 queries x 1000 reps)
  Linear scan:    1.5505 ms  (baseline)
  Python B+Tree:  0.0060 ms  (258.4x faster than linear)
  SQLite index:   0.1360 ms  (11.4x faster than linear)


In [18]:
# Range scan benchmark: SQLite idx_gpa vs Python B+ Tree vs forced linear scan
ranges = [(2.0, 2.3), (2.5, 2.8), (3.0, 3.3), (3.5, 3.8), (3.7, 4.0)]

print("Range Scan Benchmark\n")
print(f"{'GPA Range':<14} {'Linear (ms)':>12} {'B+Tree (ms)':>12} {'SQLite (ms)':>12} {'Rows':>6}")
print("-" * 62)

for low, high in ranges:
    start = time.time()
    cursor.execute("SELECT * FROM students WHERE gpa + 0 BETWEEN ? AND ?", (low, high))
    linear_rows = cursor.fetchall()
    t_linear = (time.time() - start) * 1000

    start = time.time()
    bpt_rows = bpt_gpa.range_scan(low, high)
    t_bptree = (time.time() - start) * 1000

    start = time.time()
    cursor.execute("SELECT * FROM students WHERE gpa BETWEEN ? AND ?", (low, high))
    sqlite_rows = cursor.fetchall()
    t_sqlite = (time.time() - start) * 1000

    label = f"{low}-{high}"
    print(f"{label:<14} {t_linear:>12.4f} {t_bptree:>12.4f} {t_sqlite:>12.4f} {len(sqlite_rows):>6}")


Range Scan Benchmark

GPA Range       Linear (ms)  B+Tree (ms)  SQLite (ms)   Rows
--------------------------------------------------------------
2.0-2.3              9.0787       0.0000       0.0000   1499
2.5-2.8              6.7840       0.0000       8.2169   1519
3.0-3.3              8.3797       0.0000       0.0000   1635
3.5-3.8              8.5819       0.0000       8.4753   1562
3.7-4.0              8.3296       0.0000       0.0000   1548


In [19]:
# Verify B+ Tree returns correct results
test_id = 5000
sqlite_result = cursor.execute(
    "SELECT * FROM students WHERE student_id = ?", (test_id,)
).fetchone()
bpt_result = bpt_id.search(test_id)

print("Point lookup verification:")
print(f"  SQLite:  {sqlite_result}")
print(f"  B+Tree:  {bpt_result}")
print(f"  Match:   {sqlite_result == bpt_result}")

# Range scan correctness check
sqlite_ids = sorted(
    r[0] for r in cursor.execute(
        "SELECT student_id FROM students WHERE gpa BETWEEN 3.5 AND 3.8"
    ).fetchall()
)
bpt_ids = sorted(r[0] for r in bpt_gpa.range_scan(3.5, 3.8))

print(f"\nRange scan (gpa 3.5-3.8):")
print(f"  SQLite rows: {len(sqlite_ids)}")
print(f"  B+Tree rows: {len(bpt_ids)}")
print(f"  Results match: {sqlite_ids == bpt_ids}")


Point lookup verification:
  SQLite:  (5000, 'Qewdok Qyjofguo', 2.34)
  B+Tree:  (5000, 'Qewdok Qyjofguo', 2.34)
  Match:   True

Range scan (gpa 3.5-3.8):
  SQLite rows: 1562
  B+Tree rows: 1562
  Results match: True


## Final Comparison

| Method | Complexity | Speed | Notes |
|---|---|---|---|
| Linear scan | O(n) | Slowest | Reads every row |
| Python B+ Tree | O(log n) | Fast | Pure Python overhead |
| SQLite index | O(log n) | Fast | Same algorithm in C, slightly slower due to SQL parsing |

The Python B+ tree confirms B+ tree theory works. The gap vs SQLite is due to language overhead and sql parsing, both have the same tree height (\~2 levels for 10,000 records at order=100).

## B+Tree (Order 4) Visualization for 16 rows

In [22]:
def print_tree(self):
    if not self.root.keys:
        print("(empty tree)")
        return

    print(f"\n{'='*60}")
    print(f"B+ Tree (order={self.order})")
    print(f"{'='*60}")

    # BFS level-order traversal
    from collections import deque
    queue = deque([(self.root, 0, "root")])
    current_level = 0
    level_buffer = []

    while queue:
        node, level, label = queue.popleft()

        if level != current_level:
            # Print the buffered level
            print(f"\nLevel {current_level}:")
            print("  " + "     ".join(level_buffer))
            level_buffer = []
            current_level = level

        node_type = "L" if node.is_leaf else "I"
        keys_str = "|".join(str(k) for k in node.keys)
        level_buffer.append(f"[{node_type}: {keys_str}]")

        if not node.is_leaf:
            for i, child in enumerate(node.values):
                queue.append((child, level + 1, f"child{i}"))

    # Print final level
    if level_buffer:
        print(f"\nLevel {current_level}:")
        print("  " + "     ".join(level_buffer))

    # Print leaf linked list
    print(f"\n{'─'*60}")
    print("Leaf chain (linked list):")
    leaf = self.root
    while not leaf.is_leaf:
        leaf = leaf.values[0]

    chain_parts = []
    while leaf:
        pairs = ", ".join(f"{k}:{v}" for k, v in zip(leaf.keys, leaf.values))
        chain_parts.append(f"[{pairs}]")
        leaf = leaf.next
    print("  " + " → ".join(chain_parts))
    print(f"{'='*60}\n")

# Attach to class
BPlusTree.print_tree = print_tree

In [23]:
bpt_id_test = BPlusTree(order=4)
for sid, name, gpa in all_students[:16]:
    bpt_id_test.insert(sid, (sid, name, gpa))
    bpt_id_test.print_tree()


B+ Tree (order=4)

Level 0:
  [L: 1]

────────────────────────────────────────────────────────────
Leaf chain (linked list):
  [1:(1, 'Slbrvk Fmnsknkq', 2.26)]


B+ Tree (order=4)

Level 0:
  [L: 1|2]

────────────────────────────────────────────────────────────
Leaf chain (linked list):
  [1:(1, 'Slbrvk Fmnsknkq', 2.26), 2:(2, 'Jqmrps Xuvuncto', 3.34)]


B+ Tree (order=4)

Level 0:
  [L: 1|2|3]

────────────────────────────────────────────────────────────
Leaf chain (linked list):
  [1:(1, 'Slbrvk Fmnsknkq', 2.26), 2:(2, 'Jqmrps Xuvuncto', 3.34), 3:(3, 'Aajasb Isjnuziy', 2.38)]


B+ Tree (order=4)

Level 0:
  [L: 1|2|3|4]

────────────────────────────────────────────────────────────
Leaf chain (linked list):
  [1:(1, 'Slbrvk Fmnsknkq', 2.26), 2:(2, 'Jqmrps Xuvuncto', 3.34), 3:(3, 'Aajasb Isjnuziy', 2.38), 4:(4, 'Tmvqfy Ziwehtji', 3.57)]


B+ Tree (order=4)

Level 0:
  [I: 3]

Level 1:
  [L: 1|2]     [L: 3|4|5]

────────────────────────────────────────────────────────────
Leaf chain (